In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q scipy
!pip install -q h5py
!pip install -q matplotlib
!pip install -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117
!pip install -q transformers
!pip install -q fuzzy_match
!pip install -q nltk
!pip install -q rouge
!pip install -q diffusers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 55.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 29.4 MB/s eta 0:00:00


In [ ]:
%cd /content/drive/MyDrive/Research/FINAL/Code/MMMM

# CODE

In [ ]:
import sys

sys.path.append("./models")
sys.path.append("./training")
sys.path.append("./testing")
sys.path.append("./utils")
sys.path.append("./ZuCo")

from master_init import *
from DSG import *
from load_data import *
from data import *
from dataloader import *

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter

from transformers import BartTokenizer

## Train One Epoch

In [ ]:
def train_one_epoch(dataloader, model, optimizer, criterion, tokenizer):
  results = {}
  for phase in ['train', 'dev']:
    if phase == 'train':
      model.train()  # Set model to training mode
    else:
      model.eval()   # Set model to evaluate mode

    running_loss = 0.0
    tot_cnt = 0

    # Iterate over data.
    current_data = dataloader[phase].load_data()
    while not current_data["reset"]:
      input_embeddings, seq_len, input_masks, input_mask_invert, target_ids, target_mask, sentiment_labels, sent_level_EEG = current_data["data"]

      input_embeddings_batch = input_embeddings.to(device).float()
      input_masks_batch = input_masks.to(device)
      input_mask_invert_batch = input_mask_invert.to(device)
      target_ids_batch = target_ids.to(device)

      target_ids_batch[target_ids_batch == tokenizer.pad_token_id] = -100

      optimizer.zero_grad()

      args_dict = {
        "input_data_batch" : input_embeddings_batch,
        "input_masks_batch" : input_masks_batch,
        "input_masks_invert" : input_mask_invert_batch,
        "target_ids_batch" : target_ids_batch
        }

      seq2seqLMoutput = model(
        mode="EEG-TEXT-BART",
        args_dict=args_dict
        )

      loss = seq2seqLMoutput.loss # Use the BART language modeling loss

      # Backward + Optimize only if in training phase
      if phase == 'train':
          loss.backward()
          optimizer.step()

      # Compute stats
      running_loss += loss.item() * input_embeddings_batch.size()[0]
      tot_cnt += input_embeddings_batch.size()[0]
      current_data = dataloader[phase].load_data()

    epoch_loss = running_loss / tot_cnt

    results[f"{phase}_loss"] = epoch_loss
  results["model"] = model
  return results

## Execution

In [ ]:
ZuCo_data["data"]

{'train': <data.ZuCo_dataset at 0x7cb2bef00610>,
 'dev': <data.ZuCo_dataset at 0x7cb21c6ef400>,
 'test': <data.ZuCo_dataset at 0x7cb217071720>}

In [ ]:
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

ZuCo_data = load_txt_data("./data/ZuCo")
master_eeg, master_embeds = ZuCo_data["data"], ZuCo_data["targets"]
# del ZuCo_data
ZuCo_dataloader = {
  "train" : ZuCoDataloader(master_eeg["train"], master_embeds["train"], bsz=256, drop_last=True),
  "dev" : ZuCoDataloader(master_eeg["dev"], master_embeds["dev"], bsz=256, drop_last=True),
  "test" : ZuCoDataloader(master_eeg["test"], master_embeds["test"], bsz=256, drop_last=True)
}
# del master_eeg, master_embeds

# log_dir = "./logs/EEG-TXT-BART"
# writer = SummaryWriter(log_dir=log_dir)

In [ ]:
model = INITIALIZE_MODEL(device="cuda").to(device)
learning_rate = 5e-4
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [ ]:
dsg_tasks = DSGTasks()
dsg_tasks.add_task(
  DSGTask(
    task_name="EEG-TEXT-BART",
    dataloader=ZuCo_dataloader,
    converge_lim=10,
    converge_threshold=0.0005,
    div_threshold=0.01
    )
  )

In [ ]:
import time
epoch_num = 0
print(f"|Epoch Num   |Task Name   |Current Loss      |Test Loss         |Status      |Time          |")
while epoch_num < 10:
  for task in dsg_tasks.tasks:
    start_time = time.time()
    if task.name == "EEG-TEXT-BART":
      results = train_one_epoch(
        model=model,
        dataloader=task.dataloader,
        optimizer=optimizer,
        criterion=criterion,
        tokenizer=tokenizer
        )
    model = results["model"]
    train_loss = results["train_loss"]
    dev_loss = results["dev_loss"]
    end_time = time.time()
    elapsed_time = end_time - start_time

    if task.is_converged():
      print(f"|{epoch_num:12}|{task.name:12}|{cur_loss:18.9f}|{test_loss:18.9f}|CONVERGED   |{elapsed_time:12.6f} s|")
    elif task.is_diverged():
      print(f"|{epoch_num:12}|{task.name:12}|{cur_loss:18.9f}|{test_loss:18.9f}|DIVERGED    |{elapsed_time:12.6f} s|")
    else:
      print(f"|{epoch_num:12}|{task.name:12}|{cur_loss:18.9f}|{test_loss:18.9f}|TRAINING    |{elapsed_time:12.6f} s|")

    # writer.add_scalar(f"{task.name} Train Loss", train_loss, epoch_num)
    # writer.add_scalar(f"{task.name} Dev Loss", dev_loss, epoch_num)

    task.update(epoch_num, dev_loss)
  epoch_num += 1

|Epoch Num   |Task Name   |Current Loss      |Test Loss         |Status      |Time          |
FCN OKAY
EEG OKAY


RuntimeError: ignored

# File

In [ ]:
%cd /content/drive/MyDrive/Research/FINAL/Code/MMMM
!python3 training/train_EEG-BART.py

/content/drive/MyDrive/Research/FINAL/Code/MMMM
[INFO] Loaded tokenizer.
[INFO] Prepared dataloader.
2023-11-05 14:29:57.511076: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-11-05 14:29:57.511129: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-11-05 14:29:57.511157: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-11-05 14:29:58.954882: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
[INFO] Initialized model.
[INFO] Initialized DSG.
|Epoch Num   |Task Name   |Current Loss      |Test Loss         |Status      |Time     